In [15]:
import geant4_pybind as g4
import numpy as np

ModuleNotFoundError: No module named 'geant4_pybind'

In [ ]:
class Detector(g4.G4VUserDetectorConstruction):
    def __init__(self, R=2.0 * g4.m, N=40, r=0.1 * g4.m, a=0.5,
                 half_thickness=0.01 * g4.m, rng=None):
        super().__init__()
        self.R, self.N, self.r, self.a = R, N, r, a
        self.h = half_thickness
        self.abs_length = 8.0 * g4.m
        self.rng = np.random.default_rng() if rng is None else rng
        if not isinstance(N, (int, np.integer)) or N < 1:
            raise ValueError("N должно быть положительным целым числом.")
        if not (0 < r < R and self.h > 0 and a >= 0):
            raise ValueError("Требуются 0 < r < R, half_thickness > 0, a >= 0.")

        # Весь цилиндр должен лежать внутри сферы, включая внешний обод.
        self.gap = 1.0 * g4.um
        self.center_radius = np.sqrt(R**2 - r**2) - self.h - self.gap
        self.front_radius = self.center_radius - self.h
        if self.front_radius <= 0:
            raise ValueError("Толщина ФЭУ слишком велика для этого детектора.")

        i = np.arange(N)
        z = 1 - 2 * (i + 0.5) / N
        phi = i * np.pi * (3 - np.sqrt(5))
        rho = np.sqrt(1 - z**2)
        self.axes = np.column_stack((rho * np.cos(phi), rho * np.sin(phi), z))
        self.fmt_positions = self.center_radius * self.axes

        distances = np.linalg.norm(
            self.fmt_positions[:, None, :] - self.fmt_positions[None, :, :], axis=2
        )
        np.fill_diagonal(distances, np.inf)
        self.min_distance = float(distances.min())

        if self.min_distance <= 2 * np.hypot(r, self.h):
            raise ValueError("ФЭУ расположены слишком близко. Уменьшите N или r.")

    def generate_fe(self):
        return int(self.rng.poisson(self.a))

    def Construct(self):
        nist = g4.G4NistManager.Instance()
        energies = g4.G4doubleVector([2.0 * g4.eV, 4.0 * g4.eV])

        scint = g4.G4Material("Pseudocumene", 0.876 * g4.g / g4.cm3, 2)
        scint.AddElement(nist.FindOrBuildElement("C"), 9)
        scint.AddElement(nist.FindOrBuildElement("H"), 12)
        scint_mpt = g4.G4MaterialPropertiesTable()
        scint_mpt.AddProperty("RINDEX", energies, g4.G4doubleVector([1.5, 1.5]))
        scint_mpt.AddProperty(
            "ABSLENGTH", energies, g4.G4doubleVector([self.abs_length] * 2)
        )
        scint.SetMaterialPropertiesTable(scint_mpt)

        glass = nist.FindOrBuildMaterial("G4_Pyrex_Glass")
        glass_mpt = g4.G4MaterialPropertiesTable()
        # Согласуем показатели, чтобы не отражать фотон перед регистрацией.
        glass_mpt.AddProperty("RINDEX", energies, g4.G4doubleVector([1.5, 1.5]))
        glass.SetMaterialPropertiesTable(glass_mpt)

        air = nist.FindOrBuildMaterial("G4_AIR")
        air_mpt = g4.G4MaterialPropertiesTable()
        air_mpt.AddProperty("RINDEX", energies, g4.G4doubleVector([1.0, 1.0]))
        air.SetMaterialPropertiesTable(air_mpt)

        world_solid = g4.G4Box("World", 1.5 * self.R, 1.5 * self.R, 1.5 * self.R)
        self.world_log = g4.G4LogicalVolume(world_solid, air, "World")
        self.world_phys = g4.G4PVPlacement(
            None, g4.G4ThreeVector(), self.world_log, "World", None, False, 0
        )

        sphere = g4.G4Orb("Sphere", self.R)
        self.scint_log = g4.G4LogicalVolume(sphere, scint, "Sphere")
        self.scint_phys = g4.G4PVPlacement(
            None, g4.G4ThreeVector(), self.scint_log, "Sphere", self.world_log, False, 0
        )

        # Как в задаче 3, внешняя граница поглощает фотоны, не возвращая их назад.
        self.black_surface = g4.G4OpticalSurface(
            "AbsorbingBoundary", g4.G4OpticalSurfaceModel.unified,
            g4.G4OpticalSurfaceFinish.polished, g4.G4SurfaceType.dielectric_metal
        )
        black_mpt = g4.G4MaterialPropertiesTable()
        black_mpt.AddProperty("REFLECTIVITY", energies, g4.G4doubleVector([0.0, 0.0]))
        black_mpt.AddProperty("EFFICIENCY", energies, g4.G4doubleVector([0.0, 0.0]))
        self.black_surface.SetMaterialPropertiesTable(black_mpt)
        self.black_border = g4.G4LogicalBorderSurface(
            "SphereToWorld", self.scint_phys, self.world_phys, self.black_surface
        )

        fmt_solid = g4.G4Tubs("FMT", 0, self.r, self.h, 0, 360 * g4.deg)
        self.fmt_log = g4.G4LogicalVolume(fmt_solid, glass, "FMT")
        self.fmt_volumes = []
        for k, (axis, position) in enumerate(zip(self.axes, self.fmt_positions)):
            # Активный поворот локальной оси z в радиальное направление.
            rotation = g4.G4RotationMatrix()
            rotation.rotateY(float(np.arccos(axis[2])))
            rotation.rotateZ(float(np.arctan2(axis[1], axis[0])))
            transform = g4.G4Transform3D(
                rotation, g4.G4ThreeVector(*map(float, position))
            )
            volume = g4.G4PVPlacement(
                transform, self.fmt_log, f"FMT_{k}", self.scint_log, False, k
            )
            self.fmt_volumes.append(volume)

        self.geometry_ok = not self.scint_phys.CheckOverlaps(2000, 0.0, False)
        for volume in self.fmt_volumes:
            if volume.CheckOverlaps(2000, 0.0, False):
                self.geometry_ok = False
        if not self.geometry_ok:
            raise RuntimeError("Geant4 обнаружил пересечение объемов.")
        return self.world_phys

In [ ]:
class Generator(g4.G4VUserPrimaryGeneratorAction):
    def __init__(self, position=(0.0, 0.0, 0.0), rng=None):
        super().__init__()
        self.rng = np.random.default_rng() if rng is None else rng
        self.gun = g4.G4ParticleGun(1)
        self.gun.SetParticleDefinition(g4.G4OpticalPhoton.OpticalPhotonDefinition())
        self.gun.SetParticleEnergy(3.0 * g4.eV)
        self.gun.SetParticlePosition(g4.G4ThreeVector(*map(float, position)))

    def GeneratePrimaries(self, event):
        costh = self.rng.uniform(-1, 1)
        phi = self.rng.uniform(0, 2 * np.pi)
        sinth = np.sqrt(1 - costh**2)
        n = np.array([sinth * np.cos(phi), sinth * np.sin(phi), costh])
        self.gun.SetParticleMomentumDirection(g4.G4ThreeVector(*map(float, n)))

        reference = np.array([0.0, 0.0, 1.0]) if abs(costh) < 0.9 else np.array([1.0, 0.0, 0.0])
        e1 = np.cross(n, reference)
        e1 /= np.linalg.norm(e1)
        e2 = np.cross(n, e1)
        alpha = self.rng.uniform(0, 2 * np.pi)
        polarization = np.cos(alpha) * e1 + np.sin(alpha) * e2
        self.gun.SetParticlePolarization(g4.G4ThreeVector(*map(float, polarization)))
        self.gun.GeneratePrimaryVertex(event)

In [ ]:
class Stepping(g4.G4UserSteppingAction):
    def __init__(self, detector):
        super().__init__()
        self.detector = detector
        self.hits = np.zeros(detector.N, dtype=int)
        self.photoelectrons = np.zeros(detector.N, dtype=int)
        self.bulk_absorbed = 0
        self.boundary_absorbed = 0

    def UserSteppingAction(self, step):
        track = step.GetTrack()
        if track.GetDefinition().GetParticleName() != "opticalphoton":
            return
        pre_touchable = step.GetPreStepPoint().GetTouchable()
        post_point = step.GetPostStepPoint()
        post_touchable = post_point.GetTouchable()
        pre = pre_touchable.GetVolume() if pre_touchable is not None else None
        post = post_touchable.GetVolume() if post_touchable is not None else None

        # Регистрируем именно вход в стеклянный цилиндр и сразу убиваем трек.
        if post is not None and post.GetLogicalVolume() == self.detector.fmt_log:
            fmt_id = post.GetCopyNo()
            self.hits[fmt_id] += 1
            self.photoelectrons[fmt_id] += self.detector.generate_fe()
            track.SetTrackStatus(g4.G4TrackStatus.fStopAndKill)
            return

        if pre == self.detector.scint_phys and post == self.detector.world_phys:
            self.boundary_absorbed += 1
            track.SetTrackStatus(g4.G4TrackStatus.fStopAndKill)
            return

        process = post_point.GetProcessDefinedStep()
        if process is not None and process.GetProcessName() == "OpAbsorption":
            self.bulk_absorbed += 1

In [ ]:
def expected_charge(detector, n_photons):
    """Оценки только для центрального изотропного источника."""
    d = detector.front_radius
    r = detector.r
    absorption_length = detector.abs_length
    # Приближение из задач 1 и 3, дополненное поглощением в объеме.
    p_approx = detector.N * r**2 / (4 * detector.R**2) * np.exp(-detector.R / absorption_length)

    # Точная геометрия передних круговых торцов цилиндров: L = d / cos(theta).
    mu_min = d / np.hypot(d, r)
    nodes, weights = np.polynomial.legendre.leggauss(64)
    mu = (1 + mu_min) / 2 + (1 - mu_min) * nodes / 2
    integral = (1 - mu_min) / 2 * np.dot(weights, np.exp(-d / (absorption_length * mu)))
    p_hit = detector.N * integral / 2
    mean = n_photons * p_hit * detector.a
    variance = n_photons * (p_hit * detector.a + p_hit * (1 - p_hit) * detector.a**2)
    return n_photons * detector.a * p_approx, mean, np.sqrt(variance), p_hit

In [ ]:

seed = 42
source_seed, response_seed = np.random.SeedSequence(seed).spawn(2)
g4.G4Random.setTheSeed(seed)
N_photons = 10000
source_position = np.array([0.0, 0.0, 0.0]) * g4.m

run_mgr = g4.G4RunManager()
detector = Detector(rng=np.random.default_rng(response_seed))
if np.linalg.norm(source_position) >= detector.R:
    raise ValueError("Источник должен находиться внутри сферы.")
delta = source_position - detector.fmt_positions
axial = np.sum(delta * detector.axes, axis=1)
radial_squared = np.sum(delta**2, axis=1) - axial**2
if np.any((np.abs(axial) <= detector.h) & (radial_squared <= detector.r**2)):
    raise ValueError("Источник должен быть в сцинтилляторе, а не внутри ФЭУ.")
run_mgr.SetUserInitialization(detector)

physics = g4.FTFP_BERT()
physics.SetVerboseLevel(0)
optical = g4.G4OpticalPhysics()
physics.RegisterPhysics(optical)
run_mgr.SetUserInitialization(physics)
g4.G4EmParameters.Instance().SetVerbose(0)
g4.G4HadronicParameters.Instance().SetVerboseLevel(0)

generator = Generator(source_position, np.random.default_rng(source_seed))
stepping = Stepping(detector)
run_mgr.SetUserAction(generator)
run_mgr.SetUserAction(stepping)
run_mgr.Initialize()


g4.G4Random.setTheSeed(seed)
run_mgr.BeamOn(N_photons)

total_hits = int(stepping.hits.sum())
total_charge = int(stepping.photoelectrons.sum())
accounted_photons = total_hits + stepping.bulk_absorbed + stepping.boundary_absorbed
if accounted_photons != N_photons:
    raise RuntimeError("Не сошелся баланс числа фотонов.")

print("\nРЕЗУЛЬТАТ МОДЕЛИРОВАНИЯ")
print(f"Geant4: {g4.G4Version}")
print(f"Seed: {seed}")
print(f"Проверка перекрытий: {'пройдена' if detector.geometry_ok else 'ошибка'}")
print(f"Минимальное расстояние между центрами ФЭУ: {detector.min_distance / g4.m:.6f} м")
print(f"Расстояние до передних торцов ФЭУ: {detector.front_radius / g4.m:.6f} м")
print(f"Испущено фотонов: {N_photons}")
print(f"Поглощено в объеме: {stepping.bulk_absorbed}")
print(f"Поглощено на внешней границе: {stepping.boundary_absorbed}")
print(f"Попало на ФЭУ: {total_hits}")
print(f"Баланс фотонов: {accounted_photons} / {N_photons}")
print(f"Суммарный заряд: {total_charge} фотоэлектронов")
if np.linalg.norm(source_position) == 0:
    Q_approx, Q_expected, sigma_Q, p_hit = expected_charge(detector, N_photons)
    print(f"Приближенное ожидание (ФЭУ на сфере): {Q_approx:.2f}")
    print(f"Ожидаемое число попаданий для цилиндров: {N_photons * p_hit:.2f}")
    print(f"Ожидаемый заряд для цилиндров: {Q_expected:.2f}")
    print(f"Стандартное отклонение заряда: {sigma_Q:.2f}")
    if sigma_Q > 0:
        print(f"Отклонение: {(total_charge - Q_expected) / sigma_Q:.2f} sigma")
else:
    print("Центральная аналитическая оценка неприменима к смещенному источнику.")

In [ ]:
from IPython.display import Markdown, display

rows = [
    "| Величина | Результат | Ожидание |",
    "|---|---:|---:|",
    f"| Число фотонов | {N_photons} | {N_photons} |",
    f"| Поглощено в объеме | {stepping.bulk_absorbed} | — |",
    f"| Поглощено на внешней границе | {stepping.boundary_absorbed} | — |",
    f"| Попало на ФЭУ | {total_hits} | {N_photons * p_hit:.2f} |" if np.linalg.norm(source_position) == 0 else f"| Попало на ФЭУ | {total_hits} | — |",
    f"| Заряд, фотоэлектроны | {total_charge} | {Q_expected:.2f} |" if np.linalg.norm(source_position) == 0 else f"| Заряд, фотоэлектроны | {total_charge} | — |",
]
display(Markdown("\n".join(rows)))
if np.linalg.norm(source_position) == 0 and sigma_Q > 0:
    print(f"Отклонение заряда от ожидания: {(total_charge - Q_expected) / sigma_Q:.2f} sigma.")
print(f"Баланс фотонов сошелся: {accounted_photons == N_photons}.")